# Units, neighbours, clusters and variance

**Optional code walkthrough for AI 101 (Module 9).** Not graded. Quiz scores do not depend on it.

**Objective.** See three ideas from Module 9 in short examples. Distance-based methods depend on feature units. A k-means cluster count is your choice. PCA ranks directions by variance, not by usefulness.

**Supports.** Lesson 9.1 (section “Unit scaling and method selection”), Lesson 9.3 (sections “Unsupervised grouping and compression” and “Feature scaling implications”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

Step 1 compares nearest neighbours for the same points, with a length written in metres and then in millimetres. Steps 2 and 3 use six invented operating profiles, each with a load in kW and a vibration level in mm/s.

All data are synthetic. The groups below are not fault classes.

```
Step 1 (length in m, vibration in mm/s):
  A = (0.10, 5.0)   B = (0.50, 2.1)   query = (0.12, 2.0)
Steps 2 and 3 (load in kW, vibration in mm/s):
  (10, 1.0) (12, 1.2) (11, 1.1) (40, 3.0) (42, 3.2) (41, 3.1)
```


## Step 1. Changing units changes the nearest neighbour

Lesson 9.1 says kNN uses distances in feature space. Changing a length from metres to millimetres multiplies its number by 1,000. The physical situation is the same, but the nearest neighbour can change. Here “distance” is the straight-line distance: square the differences, add them and take the square root.

In [ ]:
# Step 1: the same three points, with the length in m and then in mm
def distance(p, q):
    return ((p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2) ** 0.5

def nearest(points, query):
    best_name = None
    best_distance = None
    for name, x, y in points:
        d = distance((x, y), query)
        if best_distance is None or d < best_distance:
            best_name = name
            best_distance = d
    return best_name

# features: (length, vibration in mm/s)
points_m = [("A", 0.10, 5.0), ("B", 0.50, 2.1)]
query_m = (0.12, 2.0)
points_mm = [("A", 100.0, 5.0), ("B", 500.0, 2.1)]
query_mm = (120.0, 2.0)

print("Nearest neighbour with length in m: ", nearest(points_m, query_m))
print("Nearest neighbour with length in mm:", nearest(points_mm, query_mm))

**Expected output**

```
Nearest neighbour with length in m:  B
Nearest neighbour with length in mm: A
```

The two runs describe the same physical points. Only the unit of one feature changed, and the answer changed. In millimetres, the length differences are about 1,000 times larger and dominate the distance. Distance-based methods need scaling that respects the physical units (Lesson 9.3).

## Step 2. PCA ranks by variance, not by usefulness

PCA finds directions that capture the most variance in the data (Lesson 9.3). The `explained_variance_ratio_` values show the share of the total variance in each direction. Here the load changes much more than the vibration in these units, so the first direction is almost all the variance. High variance does not mean the direction is useful for finding a fault.

In [ ]:
# Step 2: PCA on six operating profiles (synthetic)
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

profiles = np.array([
    [10, 1.0], [12, 1.2], [11, 1.1],      # lower load
    [40, 3.0], [42, 3.2], [41, 3.1],      # higher load
])                                         # columns: load in kW, vibration in mm/s

pca = PCA()
pca.fit(profiles)
ratios = pca.explained_variance_ratio_
print(f"Share of variance, direction 1: {ratios[0]:.4f}")
print(f"Share of variance, direction 2: {ratios[1]:.4f}")

**Expected output**

```
Share of variance, direction 1: 1.0000
Share of variance, direction 2: 0.0000
```

The first direction holds almost all the variance, because the load differs by about 30 kW and the vibration by about 2 mm/s. PCA does not know which variable matters for a fault. A rare fault could sit in a low-variance direction and be thrown away. The ratios are dimensionless.

## Step 3. k-means: you choose the number of clusters

k-means groups unlabelled points around centres (Lesson 9.3). You tell it how many groups to make. Here you ask for two. The sizes are sorted, because k-means may number its groups in any order.

In [ ]:
# Step 3: k-means with k = 2 on the same profiles
k = 2
kmeans = KMeans(n_clusters=k, n_init=10, random_state=0)
kmeans.fit(profiles)

sizes = np.bincount(kmeans.labels_)     # how many points are in each group
print("Group sizes:", sorted(sizes.tolist()))

**Expected output**

```
Group sizes: [3, 3]
```

The two groups match the lower-load and higher-load profiles, three points each. These groups are mathematical groups, not fault classes. They could be two load regimes or two machine types. Only measurement context and independent evidence can say what they mean (Lesson 9.3).

## Try it

Change a value and observe. In the Step 3 code, change `k = 2` to `k = 3`. Then run Step 3 again.

**Expected result**

```
Group sizes: [1, 2, 3]
```

The algorithm splits one of the natural groups, because you asked for three. Which group it splits can depend on the starting point. It does not warn you, although the points do not form three natural groups. The number k is your choice and does not show how many real machine states exist. Lesson 9.3 suggests comparing several values of k, checking stability and checking the physical meaning.

## Limits

- Six invented points show the ideas. They are not a test of the methods. The lessons do not ask you to implement kNN, PCA or k-means.
- The walkthrough does not compare methods. Lesson 9.3 requires a fair comparison: preprocessing fitted inside each training fold, tuning on validation folds and one locked final test.
- Decision trees, random forests, boosting, logistic regression and SVMs are not shown. Lessons 9.1 and 9.2 ask you to recognize them, not to implement them.
- The group sizes for k = 2 and k = 3 were the same for 50 different random starts in a check. Which points share a group for k = 3 can still change with the start.

